# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install --quiet matplotlib numpy pandas ipython

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
from IPython.display import HTML
import pandas as pd

**Declarations**

Set the variables below and run the generation cell.

- `in_values` is the input signal over time.
- `Kp` is the proportional gain controlling reaction to the current error.
- `Ki` is the integral gain correcting accumulated past error.
- `Kd` is the derivative gain damping based on the rate of error change.


In [ ]:
# --- Declarations: editable variables for students ---
array1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 5, 5, 5, 5, 5, 5, 4, 3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
array2 = [0, 0, 0, 10, 10, 10, 0, 0, 0, 0, 1, 2, 3, 4, 5, 5, 5, 5, 5, 5, 5, 4, 3, 2, 1, 0, 0, 0, 0, 0, -10, 0, 0, 0, 0]

# PID settings
in_values = ...
Kp = ...
Ki = ...
Kd = ...

### 1. PID Controller

A PID controller (Proportional-Integral-Derivative controller) is a control loop mechanism widely used in industrial control systems. The controller continuously calculates an error value $ e(t) $ as the difference between a desired setpoint and a measured process variable, and applies a correction based on three terms: the proportional (P), integral (I), and derivative (D) of the error. The goal of the controller is to adjust the system to minimize the error over time.

The PID control law computes a control output $ out(t) $, which is the sum of three components:

- Proportional (P): This term produces an output proportional to the current error.
  >$
  >P = K_p \cdot e(t)
  >$
  >where $ K_p $ is the proportional gain, and $ e(t) $ is the current error.

- Integral (I): This term accounts for the accumulation of past errors to eliminate steady-state error. 

  >$
  >I = K_i \cdot \int_0^t e(\tau) \, d\tau
  >$
  >where $ K_i $ is the integral gain, and the integral is the accumulation of the error over time.

- Derivative (D): This term predicts future error by calculating the rate of change of the error.
  >$
  >D = K_d \cdot \frac{d}{dt} e(t)
  >$
  >where $ K_d $ is the derivative gain, and $ \frac{d}{dt} e(t) $ is the rate of change of the error.

The PID control equation that calculates the output $ out(t) $ is the sum of these three terms:

$$
out(t) = K_p \cdot e(t) + K_i \cdot \int_0^t e(\tau) \, d\tau + K_d \cdot \frac{d}{dt} e(t)
$$

#### 1.1 Discrete-Time Approximation
In practical systems, the PID controller is often implemented in a discrete-time form. In a discrete-time sampled system, the integral and derivative terms can be approximated as follows.

Integral:
>  $
>  I \approx K_i \cdot \sum_{i=0}^{n} e(i) \cdot \Delta t
>  $

Here, $ \Delta t $ is the time interval between samples.

Derivative:
>  $
>  D \approx K_d \cdot \frac{e(n) - e(n-1)}{\Delta t}
>  $

Here, $ e(n) $ and $ e(n-1) $ are the current and previous error values, respectively.

<br>

#### 1.2 Example
Assume the following parameters:

- Proportional gain $ K_p = 1.5 $
- Integral gain $ K_i = 1.0 $
- Derivative gain $ K_d = 0.5 $
- Current error $ e(t) = 2.0 $
- Previous error $ e(t-\Delta t) = 1.5 $
- Time step $ \Delta t = 1.0 $ seconds
- Accumulated error $ \int e(t) dt = 4.0 $

The control output $ out(t) $ can be computed as:

1. Proportional term:
   $
   P = 1.5 \cdot 2.0 = 3.0
   $
   
2. Integral term:
   $
   I = 1.0 \cdot 4.0 = 4.0
   $
   
3. Derivative term:
   $
   D = 0.5 \cdot \frac{2.0 - 1.5}{1.0} = 0.25
   $

4. Total output:
   $
   out(t) = 3.0 + 4.0 + 0.25 = 7.25
   $

This is the control response $ out(t) $ based on the given error and PID parameters.

<br>

#### 1.3 Example Plot

In [ ]:
# --- PID Controller Example ---
N=len(in_values)
n = np.arange(1, N+1)
result = np.ones(N)* 3 # constant 3 for all n, no system impact
error = result - in_values

# Proportional term
P = Kp * error

# Integral term (cumulative sum of error)
I = np.zeros(N)
for i in range(1, N):
    I[i] = I[i-1] + error[i]
I = Ki * I

# Derivative term (difference of consecutive errors)
D = np.zeros(N)
for i in range(1, N):
    D[i] = error[i] - error[i-1]
D = Kd * D

out = P + I + D

def pretty_array(name, arr):
    formatted = " ".join(f"{x:6.2f}" for x in arr)
    print(f"{name}(t)    = [{formatted}]")

pretty_array("error", error)
print("\n")
pretty_array("    P", P)
pretty_array("    I", I)
pretty_array("    D", D)
pretty_array("  out", out)


fig, ax = plt.subplots(figsize=(10, 5))
theLineWidth = 3
ax.cla()
ax.axis([None, None, None, None])
props = dict(boxstyle='round', alpha=1, facecolor=(.6, 0.61, 0.8), edgecolor='w')

plt.plot(n, P, label="P(t) - Proportional", linewidth=theLineWidth)
plt.plot(n, I, label="I(t) - Integral", linewidth=theLineWidth)
plt.plot(n, D, label="D(t) - Derivative", linewidth=theLineWidth)
plt.plot(n, out, label="out(t) - PID Output", linewidth=theLineWidth)
plt.legend()
plt.xlabel("t")
plt.ylabel("PID Controller Output Terms")
plt.grid(True)
ax.set_facecolor((.2, 0.2, 0.3))
ax.legend(fancybox=True, framealpha=1,
          loc="upper left", facecolor=(.6, 0.61, 0.8), edgecolor='w')
plt.tight_layout()

### 2. PID Determinism

A PID controller is deterministic as it always produces the same output for the same input (error signal) and configuration (gains). The given set of initial conditions (e.g., the current error, previous errors, and system state) and parameters (proportional, integral, and derivative gains) and the predefined formula make the outcome predictable and repeatable.

The controller's actions at any given time depend solely on the current and past errors, along with the fixed parameters $K_p$, $K_i$, and $K_d$. As a result, for the same error conditions and controller gains, the output will always be the same.

#### 2.1 Formal Terms

A deterministic system is a system in which the future state is fully determined by the current state and inputs, without any randomness. For a PID controller, given a specific error signal $ e(t) $, the control output $ out(t) $ is calculated deterministically by the PID formula:
  $$
  out(t) = K_p \cdot e(t) + K_i \cdot \int_0^t e(\tau) \, d\tau + K_d \cdot \frac{d}{dt} e(t)
  $$

Since there is no randomness involved, and every calculation is based on deterministic mathematical operations, the PID controller is deterministic.

#### 2.2 Non-Deterministic Controllers
A non-deterministic controller, on the other hand, may involve some level of randomness or uncertainty in its decision-making process. This could be due to factors such as:

- Stochastic control: The control system is designed to handle random disturbances or uncertainty in the model, and it produces different outputs even for the same inputs.
  
- Adaptive controllers: These controllers adjust their parameters based on the system's behavior or external inputs, which can introduce non-deterministic behavior if, for example, the adaptation process is influenced by random factors.
  
- Controllers with random noise or probabilistic decisions: For instance, a controller that uses a randomized or probabilistic algorithm might apply random perturbations to explore different control strategies.

An example of a non-deterministic controller would be a reinforcement learning (RL)-based controller. In RL, the controller (or agent) learns from interaction with the environment and may incorporate randomness in its actions during the learning process (e.g., using exploration strategies like ε-greedy, where actions are chosen randomly with a probability $ \epsilon $). In this case, the output is not purely based on a fixed rule like in a PID controller, but instead on a probabilistic policy that may vary over time. Even if the same state is encountered multiple times, the RL-based controller may produce different actions due to the randomness in the exploration process. In contrast to deterministic PID control, where the same error leads to the same control action, RL controllers can produce varying actions in identical conditions, especially during the learning phase, making them non-deterministic. 

### 3. Python Implementation

In the PID controller function used below, `errors` is an array of all the errors $e$ we've recorded so far,
`Kp`, `Ki`, `Kd` are the PID constants, and `time_delta` is the amount of time between
error measurements. In Python, the three terms of the PID equation can be computed as follows. The proportional term is `Kp * current_error`, where `current_error` is the last
recorded error: `errors[-1]`. Because we there is a list of discrete error measurements, the integral is replaced by a with summation (In Python: `Ki * sum(errors) * time_delta`):
$$K_I \int e \, dt \approx K_I \sum e \Delta t$$
The derivative of the error is approximated using two consecutive error measurements $e_n$ and $e_{n-1}$, (In Python: `Kd*(current_error - prev_error) / time_delta`):
$$K_D \frac{de}{dt} \approx K_D \frac{e_n - e_{n-1}}{\Delta t}$$

In [ ]:
# --- PID Response ---
def pid_response(errors, Kp, Ki, Kd, time_delta):
    current_error = errors[-1]
    if len(errors) > 1:
        prev_error = errors[-2]
    else:
        # if there are no two error measurements, just reuse the first error
        prev_error = current_error

    proportional = Kp*current_error
    integral = Ki * sum(errors) * time_delta
    derivative = Kd*(current_error - prev_error) / time_delta

    return proportional + integral + derivative

#### 3.1 Process
The process is the system that acts upon the response given by the PID controller. The function `process` accepts a PID controller response `res` and the current state of the system `state`. The implementation is not relevant,  and used here only for illustration purposes. The error function expresses how far the current state `s` is from our desired state `d`.

In [ ]:
# --- Simple Process Model ---
p_response = 0
constant_shift = 0

def process(res, state):
    global p_response
    p_response = p_response*0.96 - 0.06*res + constant_shift
    # add "real-world" constraints
    p_response = min(0.3, p_response)
    p_response = max(-0.3, p_response)
    return state + p_response

def calc_error(s, d):
    return s - d

#### 3.2 Simulation

In [ ]:
# --- PID Simulation ---
def simulate_pid(Kp, Ki, Kd):
    time_delta = 0.02
    duration = 30
    # create a list of times from 0 to `duration` seconds with steps of `time_delta`
    # in total, duration / time_delta observations will be recorded
    times = np.arange(0, duration, time_delta)

    desired = np.zeros(times.size)
    desired[100:170] = 1

    states = np.zeros(times.size)
    states[0] = 3  # initial state

    errors = np.zeros(times.size)
    errors[0] = calc_error(states[0], desired[0])  # initial error

    for i in range(1, len(times)):
        res = pid_response(errors[:i], Kp, Ki, Kd, time_delta=time_delta)
        state = process(res, states[i - 1])
        states[i] = state
        errors[i] = calc_error(state, desired[i])

    return times, states, desired, errors

#### 3.3 Plot

The simulation might take a few minutes, before the updated plot will appear.

In [ ]:
# --- Run Simulation and Plot Results ---
fig, ax = plt.subplots(figsize=(10, 5))
theLineWidth = 3
ax.cla()
ax.axis([None, None, None, None])
ax.set_xlabel("Time")
ax.set_ylabel("PID Controller State")

line, = ax.plot([], [], linewidth=theLineWidth, label="PID Controller State", color="w")
ax.set_xlim((0, 10))
ax.set_ylim((-1, 3))
props = dict(boxstyle='round', alpha=1, facecolor=(.6, 0.61, 0.8), edgecolor='w')

tex = ax.text(0.78, 0.95, '', transform=ax.transAxes, verticalalignment='top')

Kp = 2
Kd = 0.05
Ki = 0.1
t, s, d, e = simulate_pid(Kp, Ki, Kd)
line.set_data(t, s)
fmt_str = 'Kp=%.2f\nKi=%.2f\nKd=%.2f'
s = fmt_str % (Kp,Ki,Kd)
tex.set_text(s)
tex.set_fontsize(14)
tex.set_bbox(props)

ax.plot(t, d, linewidth=theLineWidth, label="Desired State", color="green")

ax.grid(True)
ax.set_facecolor((.2, 0.2, 0.3))
ax.legend(fancybox=True, framealpha=1,
          loc="upper left", facecolor=(.6, 0.61, 0.8), edgecolor='w')
plt.tight_layout()


##### 3.2.2 Animation

This animated gif shows how the output changes with changing parameters. REMARK: Reload this notebook view in case of updates (in case of changed input parameters).


In [ ]:
# --- Animation Function ---
def animate(i):
    Kp = 1.5 - min(i, 100)*0.005
    Kd = 0
    if i > 100:
        Kd = min((i-100)/2.0, 30) * 0.001
    t, s, d, e = simulate_pid(Kp, Ki, Kd)
    line.set_data(t, s)
    fmt_str = 'Kp=%.2f\nKi=%.2f\nKd=%.2f'
    s = fmt_str % (Kp, Ki, Kd)
    tex.set_text(s)
    tex.set_bbox(props)
    return (line,)


anim = animation.FuncAnimation(fig, animate,
                               frames=250, interval=20, blit=True)

HTML(anim.to_html5_video())
anim.save('PID/pid_animation.gif')

#### 3.2 Ki and the Constant Shift

Earlier, `constant_shift` was set to be non-zero (just above the `process` function). This affects the PID controller response graph by introducing a constant error.
The `Ki` weight constant can be used to mitigate the constant shift:
- If `Ki` is set to 0, the PID controller will not be able to correct the constant shift.
- If `Ki` is set to a non-zero value, the PID controller will be able to correct the constant shift.
    
    

### 4. Open-Loop and Closed-Loop Control

Control systems operate either in open-loop or closed-loop form, depending on whether they use feedback to adjust their behavior.

**Open-loop reflexes** act immediately and predictably based on input, without checking the output or current system state. They are pre-defined reactions — direct mappings from stimulus to command — ideal for fast, simple actions where conditions are consistent and feedback is unnecessary.
Example: turning on a heater for a fixed time without measuring temperature.

**Closed-loop control** incorporates feedback to continuously correct the system toward a desired set-point. The controller measures the actual state, compares it to the target, and adjusts its output to minimize error. Closed-loop systems can compensate for disturbances, but multiple control processes acting on the same system can introduce competition and instability if not coordinated properly.


### 5 Sampling Rate and System Settling Time

Digital control systems work by sampling measurements at discrete intervals. The sampling rate ($\text{rate} = 1 / \Delta t$) directly affects control quality.

* A low sampling rate adds delay, making the system sluggish or unstable.
* A very high sampling rate can amplify noise in derivative terms or cause integrator overflow in digital implementations.

A practical rule of thumb:
Choose $\Delta t$ between one-tenth and one-hundredth of the system’s settling time.

The **settling time** is the duration required for a system’s output to enter and remain within a specified error band (often ±5%) around its final steady-state value without oscillation. It reflects how quickly the system stabilizes after an input change.


### 6 Controller Stability and Predictability

When designing a controller, stability and predictability are key considerations.

Questions to evaluate include:

* How quickly does the system respond to a change in set-point?
* Does it settle smoothly, or does it oscillate around the target?

Classical PID controllers are deterministic: each control term (proportional, integral, derivative) can be described in closed form. This allows engineers to predict system response mathematically and ensure reliable, repeatable performance.

Deterministic control algorithms are required in life-support and safety-critical domains (e.g., medical regulation under FDA standards), where behavior must be fully explainable.
Machine learning and adaptive algorithms, while promising, are not yet certified for such use due to their non-deterministic nature.
